# COMP64702 RAG Coursework
## Notebook 3: Evaluation

Evaluates the RAG system outputs against gold-standard benchmark answers using:
- **ROUGE-1, ROUGE-2, ROUGE-L** — n-gram overlap with gold answers
- **BLEU** — precision-focused n-gram overlap
- **BERTScore** — semantic similarity via contextual embeddings
- **Faithfulness** — token overlap between response and retrieved context (grounding check)

Also compares **hybrid retrieval** vs **dense-only baseline** to demonstrate improvement.

In [1]:
# Install dependencies (run once)
!pip install rouge-score bert-score nltk -q

In [2]:
# Imports
import json
import re
import numpy as np
from pathlib import Path

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score
import nltk
nltk.download('punkt', quiet=True)

print('Imports OK')

/opt/miniconda3/envs/rag-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [3]:
# File paths
DATA_DIR        = Path('.')
benchmark_path  = DATA_DIR / 'benchmark_dataset.json'
outputs_path    = DATA_DIR / 'test_outputs.json'
baseline_path   = DATA_DIR / 'test_outputs_baseline.json'
eval_path       = DATA_DIR / 'evaluation_results.json'

with open(benchmark_path, 'r', encoding='utf-8') as f:
    benchmark = json.load(f)

with open(outputs_path, 'r', encoding='utf-8') as f:
    outputs = json.load(f)

print(f'Benchmark   : {len(benchmark)} items')
print(f'Predictions : {len(outputs["results"])} items')

Benchmark   : 100 items
Predictions : 100 items


In [4]:
# Match benchmark and outputs by query_id
gold_map = {item['query_id']: item for item in benchmark}
pred_map = {item['query_id']: item for item in outputs['results']}

common_ids = sorted(set(gold_map) & set(pred_map))
print(f'Matched query IDs: {len(common_ids)}')

predictions = [pred_map[qid]['response']           for qid in common_ids]
references  = [gold_map[qid]['answer']             for qid in common_ids]
contexts    = [pred_map[qid].get('retrieved_context', []) for qid in common_ids]

Matched query IDs: 100


In [5]:
# ── Metric functions ──────────────────────────────────────────────────────

rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def rouge_scores(pred, ref):
    scores = rouge.score(ref, pred)
    return {
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
    }

def bleu_score(pred, ref):
    return sentence_bleu(
        [ref.split()], pred.split(),
        smoothing_function=smooth
    )


# ── Improved faithfulness ──────────────────────────────────────────────────
# Measures what fraction of meaningful words in the response
# are grounded in the retrieved context.
#
# Improvements over naive exact-match:
#   1. Simple suffix stemming so 'marinating'/'marinated', 'cooking'/'cooked'
#      count as the same token.
#   2. Expanded stopword list to avoid penalising common function words.
#   3. Number normalisation — '5' and 'five' treated equivalently.

STOPWORDS = {
    'the','a','an','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','could',
    'should','may','might','shall','to','of','in','on','at','by',
    'for','with','about','and','or','but','if','as','it','its',
    'this','that','these','those','from','into','through','during',
    'including','until','while','per','also','both','each','more',
    'most','other','some','such','than','then','there','when',
    'where','which','who','whom','whose','how','what','very',
    'just','not','no','so','up','out','can','one','two','three',
    'used','use','uses','using','make','made','makes','known',
    'often','well','all','any','between','its','their','our',
}

NUM_WORDS = {'one':'1','two':'2','three':'3','four':'4','five':'5',
             'six':'6','seven':'7','eight':'8','nine':'9','ten':'10'}

def stem(word):
    """Very simple suffix stemmer."""
    for suffix in ('ing','tion','tions','ed','ly','ies','er','est','s'):
        if len(word) > len(suffix) + 3 and word.endswith(suffix):
            return word[:-len(suffix)]
    return word

def tokenize_for_faith(text):
    tokens = set()
    for w in re.findall(r'[a-z0-9]+', text.lower()):
        w = NUM_WORDS.get(w, w)   # normalise number words
        if w not in STOPWORDS and len(w) > 2:
            tokens.add(stem(w))   # stem before adding
    return tokens

def faithfulness(response, context_chunks):
    """Fraction of stemmed response tokens found in retrieved context."""
    ctx_text = ' '.join(c.get('text', '') for c in context_chunks)
    resp_tokens = tokenize_for_faith(response)
    ctx_tokens  = tokenize_for_faith(ctx_text)
    if not resp_tokens:
        return 0.0
    overlap = resp_tokens & ctx_tokens
    return len(overlap) / len(resp_tokens)

print('Metric functions defined')

Metric functions defined


In [6]:
# ── BERTScore (batch) ─────────────────────────────────────────────────────
print('Computing BERTScore...')
P, R, F1 = bertscore_score(predictions, references, lang='en', verbose=False)
bertscore_f = F1.tolist()
print(f'BERTScore done — mean F1: {np.mean(bertscore_f):.4f}')

Computing BERTScore...


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7453.39it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore done — mean F1: 0.8922


In [7]:
# ── Per-item evaluation ───────────────────────────────────────────────────
per_item = []

for i, qid in enumerate(common_ids):
    pred = predictions[i]
    ref  = references[i]
    ctx  = contexts[i]

    r_scores = rouge_scores(pred, ref)
    b_score  = bleu_score(pred, ref)
    f_score  = faithfulness(pred, ctx)

    per_item.append({
        'query_id':    qid,
        'rouge1_f':    r_scores['rouge1_f'],
        'rouge2_f':    r_scores['rouge2_f'],
        'rougeL_f':    r_scores['rougeL_f'],
        'bleu':        b_score,
        'bertscore_f': bertscore_f[i],
        'faithfulness': f_score,
    })

print(f'Evaluated {len(per_item)} items')

Evaluated 100 items


In [8]:
# ── Aggregate metrics ─────────────────────────────────────────────────────
aggregate = {
    'mean_rouge1_f':    float(np.mean([x['rouge1_f']    for x in per_item])),
    'mean_rouge2_f':    float(np.mean([x['rouge2_f']    for x in per_item])),
    'mean_rougeL_f':    float(np.mean([x['rougeL_f']    for x in per_item])),
    'mean_bleu':        float(np.mean([x['bleu']        for x in per_item])),
    'mean_bertscore_f': float(np.mean([x['bertscore_f'] for x in per_item])),
    'mean_faithfulness':float(np.mean([x['faithfulness']for x in per_item])),
}

print('Evaluation Results (aggregate):')
for k, v in aggregate.items():
    print(f'  {k:<25} : {v:.4f}')

Evaluation Results (aggregate):
  mean_rouge1_f             : 0.4835
  mean_rouge2_f             : 0.2114
  mean_rougeL_f             : 0.3350
  mean_bleu                 : 0.1164
  mean_bertscore_f          : 0.8922
  mean_faithfulness         : 0.5574


In [9]:
# ── Save evaluation_results.json ──────────────────────────────────────────
eval_results = {'per_item': per_item, 'aggregate': aggregate}

with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f'Saved to {eval_path}')

Saved to evaluation_results.json


## Baseline Comparison

Compare hybrid retrieval (BM25 + dense RRF) against dense-only baseline.
This demonstrates the improvement from the hybrid approach, relevant to the
**competitive retrieval performance** marking criterion.

In [10]:
# ── Baseline comparison ───────────────────────────────────────────────────
import os

if not os.path.exists(baseline_path):
    print('Baseline outputs not found — run Cell 18 in notebook 02 first.')
else:
    with open(baseline_path, 'r', encoding='utf-8') as f:
        baseline_outputs = json.load(f)

    base_pred_map = {r['query_id']: r for r in baseline_outputs['results']}
    base_common   = sorted(set(gold_map) & set(base_pred_map))

    base_preds = [base_pred_map[qid]['response'] for qid in base_common]
    base_refs  = [gold_map[qid]['answer']         for qid in base_common]
    base_ctxs  = [base_pred_map[qid].get('retrieved_context', []) for qid in base_common]

    # Compute baseline metrics
    print('Computing baseline BERTScore...')
    _, _, base_F1 = bertscore_score(base_preds, base_refs, lang='en', verbose=False)
    base_bscore = base_F1.tolist()

    base_per_item = []
    for i, qid in enumerate(base_common):
        r_scores = rouge_scores(base_preds[i], base_refs[i])
        base_per_item.append({
            'query_id':     qid,
            'rouge1_f':     r_scores['rouge1_f'],
            'rouge2_f':     r_scores['rouge2_f'],
            'rougeL_f':     r_scores['rougeL_f'],
            'bleu':         bleu_score(base_preds[i], base_refs[i]),
            'bertscore_f':  base_bscore[i],
            'faithfulness': faithfulness(base_preds[i], base_ctxs[i]),
        })

    base_agg = {
        k: float(np.mean([x[k] for x in base_per_item]))
        for k in ['rouge1_f','rouge2_f','rougeL_f','bleu','bertscore_f','faithfulness']
    }

    # Side-by-side comparison
    print()
    print(f'{"Metric":<20} {"Hybrid":>10} {"Baseline":>10} {"Delta":>10}')
    print('-' * 52)
    metric_map = [
        ('ROUGE-1',      'rouge1_f'),
        ('ROUGE-2',      'rouge2_f'),
        ('ROUGE-L',      'rougeL_f'),
        ('BLEU',         'bleu'),
        ('BERTScore',    'bertscore_f'),
        ('Faithfulness', 'faithfulness'),
    ]
    for label, key in metric_map:
        hyb  = aggregate[f'mean_{key}']
        base = base_agg[key]
        delta = hyb - base
        arrow = '▲' if delta > 0 else '▼'
        print(f'{label:<20} {hyb:>10.4f} {base:>10.4f} {arrow}{abs(delta):>8.4f}')

    # Save full results including baseline
    eval_results['baseline_aggregate'] = base_agg
    eval_results['baseline_per_item']  = base_per_item
    with open(eval_path, 'w', encoding='utf-8') as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)
    print(f'\nFull results (with baseline) saved to {eval_path}')

Computing baseline BERTScore...


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 9944.74it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Metric                   Hybrid   Baseline      Delta
----------------------------------------------------
ROUGE-1                  0.4835     0.4781 ▲  0.0054
ROUGE-2                  0.2114     0.2101 ▲  0.0013
ROUGE-L                  0.3350     0.3378 ▼  0.0028
BLEU                     0.1164     0.1162 ▲  0.0002
BERTScore                0.8922     0.8928 ▼  0.0006
Faithfulness             0.5574     0.5534 ▲  0.0040

Full results (with baseline) saved to evaluation_results.json


In [11]:
# ── Per-item breakdown — worst and best ──────────────────────────────────
sorted_by_faith = sorted(per_item, key=lambda x: x['faithfulness'])

print('── 5 lowest faithfulness ──────────────────────────────')
for item in sorted_by_faith[:5]:
    print(f'  {item["query_id"]}  '
          f'faith={item["faithfulness"]:.3f}  '
          f'rouge1={item["rouge1_f"]:.3f}  '
          f'bert={item["bertscore_f"]:.3f}')

print()
print('── 5 highest faithfulness ─────────────────────────────')
for item in sorted_by_faith[-5:]:
    print(f'  {item["query_id"]}  '
          f'faith={item["faithfulness"]:.3f}  '
          f'rouge1={item["rouge1_f"]:.3f}  '
          f'bert={item["bertscore_f"]:.3f}')

print()
faiths = [x['faithfulness'] for x in per_item]
print(f'Items faith > 0.7 : {sum(1 for f in faiths if f > 0.7)}')
print(f'Items faith > 0.5 : {sum(1 for f in faiths if f > 0.5)}')
print(f'Items faith < 0.3 : {sum(1 for f in faiths if f < 0.3)}')

── 5 lowest faithfulness ──────────────────────────────
  sa_011  faith=0.156  rouge1=0.476  bert=0.892
  sa_038  faith=0.161  rouge1=0.222  bert=0.840
  sa_010  faith=0.182  rouge1=0.442  bert=0.880
  sa_025  faith=0.186  rouge1=0.310  bert=0.868
  sa_029  faith=0.214  rouge1=0.437  bert=0.885

── 5 highest faithfulness ─────────────────────────────
  sa_077  faith=0.880  rouge1=0.516  bert=0.886
  sa_043  faith=0.907  rouge1=0.629  bert=0.911
  sa_002  faith=0.920  rouge1=0.615  bert=0.918
  sa_047  faith=0.967  rouge1=0.692  bert=0.942
  sa_060  faith=1.000  rouge1=0.440  bert=0.882

Items faith > 0.7 : 21
Items faith > 0.5 : 61
Items faith < 0.3 : 6


In [12]:
# ── Final summary ─────────────────────────────────────────────────────────
print('=' * 50)
print('  EVALUATION COMPLETE')
print('=' * 50)
print(f'  Queries evaluated : {len(per_item)}')
print()
print(f'  ROUGE-1      : {aggregate["mean_rouge1_f"]:.4f}')
print(f'  ROUGE-2      : {aggregate["mean_rouge2_f"]:.4f}')
print(f'  ROUGE-L      : {aggregate["mean_rougeL_f"]:.4f}')
print(f'  BLEU         : {aggregate["mean_bleu"]:.4f}')
print(f'  BERTScore    : {aggregate["mean_bertscore_f"]:.4f}')
print(f'  Faithfulness : {aggregate["mean_faithfulness"]:.4f}')
print('=' * 50)

  EVALUATION COMPLETE
  Queries evaluated : 100

  ROUGE-1      : 0.4835
  ROUGE-2      : 0.2114
  ROUGE-L      : 0.3350
  BLEU         : 0.1164
  BERTScore    : 0.8922
  Faithfulness : 0.5574
